File này và các file tương tự nhằm mục đích khai thác và đánh giá hiệu quả của các mô hình embedding

# Load dataset

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [42]:
from transformers import AutoTokenizer
from transformers import AutoModel
from transformers import BertTokenizer
from sentence_transformers import SentenceTransformer

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os
import json
import glob
import time
import copy

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [30]:
vietnamese_history_dataset_path = "../../datasets/vietnamese_history_dataset/data.json"
df = pd.read_json(vietnamese_history_dataset_path)
df.head()

,title,content,type
0,BUỔI ĐẦU LỊCH SỬ NƯỚC TA,Chương: BUỔI ĐẦU LỊCH SỬ NƯỚC TA,chapter
1,THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA,Chương: BUỔI ĐẦU LỊCH SỬ NƯỚC TA\nBài: THỜI NG...,lesson
2,Những dấu tích của Người tối cổ được tìm thấy ...,Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA\nNhững ...,title
3,"Ở giai đoạn đầu, Người tinh khôn sống như thế ...",Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA\nỞ giai...,title
4,Giai đoạn phát triển của Người tinh khôn có gì...,Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA\nGiai đ...,title


# Chuẩn bị dữ liệu đánh giá retrieval

## Truy vấn (query)
    Một câu hoặc một đoạn văn

In [85]:
queries = [
    "Hãy liệt kê những địa điểm tìm thấy dấu tích của người tối cổ?",
    "Vào thời Văn Lang - Âu Lạc, công cụ sản xuất được cải tiến như thế nào?",
    "Hãy kể về lịch sử nước cham-pa?",
    "Đinh Bộ Lĩnh lên ngôi đặt tên nước là gì?",
    "Nho giáo bắt đầu hình thành ở nước ta bắt đầu từ khi nào?",
    "Lịch sử Nhật Bản sau chiến tranh thế giới thứ 2",
    "Khoa học công nghệ",
]

## Corpus
    Kho dữ liệu chứa nhiều đoạn văn bản hoặc tài liệu.

## Ground Truth
    Tập hợp kết quả đúng ứng với mỗi truy vấn

In [86]:
ground_truths = [
    # query 1
    [
        {
            "title": "Đời sống vật chất",
            "content": "Bài: ĐỜI SỐNG CỦA NGƯỜI NGUYÊN THỦY TRÊN ĐẤT NƯỚC TA\nĐời sống vật chất: Trong quá trình sinh sống, người nguyên thủy thời Sơn Vi - Hòa Bình - Bắc Sơn - Hạ Long thường xuyên tìm cách cải tiến công cụ lao động. Nguyên liệu chủ yếu là đá. Ban đầu, người thời Sơn Vi chỉ biết ghè đẻo các hòn cuội ven sông làm rìu, nhưng đến thời Hòa Bình - Bắc Sơn - Hạ Long, họ đã biết mài đá, dùng nhiều loại đá khác nhau để làm công cụ các loại như rìu, bôn®, chày. Họ còn biết dùng tre, gỗ, xương, sừng làm công cụ và đồ dùng cần thiết, sau đó biết làm đồ gốm. Người nguyên thủy đã biết trồng trọt và chăn nuôi. Nguồn thức ăn ngày càng tăng lên. Ngoài cây củ kiểm được, họ còn trồng rau, đậu, bí, bầu... Ngoài thú rừng săn được, họ còn nuôi thêm chó, lợn. Người nguyên thủy sống chủ yếu ở các hang động, mái đá. Họ cũng biết làm các túp lều lớp bằng cỏ hoặc lá cây.",
        },        
    ],
    # query 2
    [
        {
            "title": "Công cụ sản xuất được cải tiến như thế nào ?",
            "content": "Bài: NHỮNG CHUYỂN BIẾN TRONG ĐỜI SỐNG KINH TẾ\nCông cụ sản xuất được cải tiến như thế nào ?: Những người nguyên thủy trên đất nước ta tiếp tục mở rộng vùng cư trú. Một số đã dừng lại ở các vùng chân núi, thung lũng ven khe, suối..., một số khác thì chuyển xuống các vùng đất bãi ven sông, dựng chòi, cuộc đất trồng trọt, làm chuồng nuôi lợn, gà, chó... Các nhà khảo cổ đã phát hiện được rất nhiều địa điểm chứa đựng những lưỡi rìu đá có vai được mài rộng ra hai mặt, những lưỡi đục, những bàn mài và những mảnh của đá. Số công cụ bằng xương, sừng cũng nhiều hơn. Bên cạnh đó, họ còn tìm thấy nhiều loại hình đồ gốm như bình, vò, nồi cùng nhiều hạt chuỗi đá, vỏ ốc... Người nguyên thuy cũng đã biết làm chì lưới bằng đất nung để đánh cá.\nTrong một số điểm như Phùng Nguyên (Phú Thọ), Hoa Lộc (Thanh Hoá), Lung Leng (Kon Tum) có niên đại cách đây 4.000 - 3.500 năm, các nhà khảo cổ đã phát hiện được hàng loạt công cụ như rìu đá, bôn đá được mài nhẵn toàn bộ, có hình dáng cân xứng. Họ còn tìm thấy những đồ trang sức, những loại đồ gốm khác nhau như bình, vò, vái, bát đĩa, cốc có chân cao... Những mảnh gốm thường in hoa văn hình chữ S nối nhau, đối xứng hoặc in những con dấu nổi, liền nhau với những đường cuộn theo hình tròn hay hình chữ nhật, những đường chấm nhỏ li ti chạy dài trên một nền phẳng.",
        },
        {
            "title": "Thuật luyện kim đã được phát minh như thế nào ?",
            "content": "Bài: NHỮNG CHUYỂN BIẾN TRONG ĐỜI SỐNG KINH TẾ\nThuật luyện kim đã được phát minh như thế nào ?: Cuộc sống của người nguyên thủy ngày càng ổn định hơn. Dần dần đã xuất hiện những làng bản đông dân ở các vùng ven sông, đặc biệt là ven các con sông lớn như sông Hồng, sông Mã, sông Cả, sông Đồng Nai..., gồm nhiều gia đình thuộc nhiều thị tộc khác nhau. Cuộc sống định cư lâu dài đòi hỏi con người lúc đó phải cải tiến hơn nửa các công cụ sản xuất và đồ dùng hàng ngày.\nNhờ sự phát triển của nghề làm đồ gốm, người Phùng Nguyên, Hoa Lộc đã phát minh ra thuật luyện kim.\nKim loại đầu tiên được dùng là đồng. Ở Phùng Nguyên, Hoa Lộc và các điểm khác cùng thời trên khắp nước ta, người ta đã phát hiện được nhiều công cụ đồng, xỉ đồng, dây đồng, dùi đồng. Thuật luyện kim đã được phát minh.",
        }
    ],
    # query 3
    [
        {
            "title": "NƯỚC CHAM-PA TỪ THẾ KỶ II ĐẾN THẾ KỶ X",
            "content": "Chương: THỜI KÌ BẮC THUỘC VÀ ĐẤU TRANH GIÀNH ĐỘC LẬP\nBài: NƯỚC CHAM-PA TỪ THẾ KỶ II ĐẾN THẾ KỶ X",
        },
        {
            "title": "Nước Cham-pa độc lập ra đời",
            "content": "Bài: NƯỚC CHAM-PA TỪ THẾ KỶ II ĐẾN THẾ KỶ X\nNước Cham-pa độc lập ra đời: Thời Hán, sau khi chiếm được Giao Chỉ và Cửu Chân, quân Hán đánh xuống phía nam chiếm cả đất của người Chăm cổ, sáp nhập vào Nhật Nam, đặt ra huyện Tường Lâm.\nQuận Nhật Nam (từ Hoành Sơn trở vào) gồm năm huyện. Huyện xa nhất là Tường Lâm (nay thuộc đất Quảng Nam, Quảng Ngãi, Bình Định), là địa bàn sinh sống của bộ lạc Dừa - tước người Chăm cổ, thuộc nền văn hóa đồng thời Sa Huỳnh khá phát triển.\nVào thế kỉ I, nhân dân Giao Châu nhiều lần nổi dậy. Nhà Hán tới ra bất lực, nhất là đối với các quận xa. Năm 192 - 193, nhân dân Tường Lâm dưới sự lãnh đạo của Khu Liên đã nổi dậy giành độc lập. Khu Liên tự xưng làm vua, đặt tên nước là Lâm Ấp.\nQuốc gia Lâm Ấp có lực lượng quân sự khá mạnh (đạo quân thông thường gồm 4 - 5 vạn người). Các vua Lâm Ấp đã hợp nhất bộ lạc Dừa với bộ lạc Cau ở phía nam, tấn công các nước láng giềng, mở rộng lãnh thổ - về phía bắc đến Hoành Sơn (huyện Tây Quận), phía nam đến Phan Rang, rồi đổi tên nước là Cham-pa (sử sách Trung Quốc gọi là nước Hoàn Vương), đóng đô ở Sin-ha-pu-ra (Trà Kiều - Quảng Nam).",
        },
        {
            "title": "Tình hình kinh tế, văn hóa Cham-pa từ thế kỉ II đến thế kỉ X",
            "content": "Bài: NƯỚC CHAM-PA TỪ THẾ KỶ II ĐẾN THẾ KỶ X\nTình hình kinh tế, văn hóa Cham-pa từ thế kỉ II đến thế kỉ X: Người Chăm biết sử dụng công cụ bằng sắt và dùng trâu, bò kéo cày. Nguồn sống chủ yếu của họ là nông nghiệp trồng lúa, mỗi năm hai vụ. Người Chăm còn làm ruộng bậc thang ở sườn đồi, núi. Họ sáng tạo ra xe gương nước để đưa nước từ sông, suối lên ruộng và từ ruộng thấp lên ruộng cao. Họ còn trồng các loại cây ăn quả Dãy Hoành Sơn (cau, dừa, mít...) và các loại cây khác (ao nhồ sẽ...), làm đồ gốm khá phát triển. Cư dân sống ven biển, ven sông có nghề đánh cá. Người Chăm thường trao đổi, buôn bán với nhân dân các đoàn phố ở. Một Cố Lấp, Trưng Quốc, còn kiêm nghề cướp biển và buôn bán nô lệ. Từ thế kỉ IV, người Chăm đã có chữ viết riêng, bắt nguồn từ chữ Phạn của người Ấn Độ. Nhân dân Chăm theo đạo Bà La Môn và đạo Phật. Người Chăm có tục hỏa táng người chết, bỏ tro vào bình hoặc vò gốm rồi ném xuống sông hay xuống biển. Họ ở nhà sàn và cũng có thói quen ăn trầu cau. Người Chăm đã sáng tạo ra một nền nghệ thuật đặc sắc, tiêu biểu là các tháp Chăm, đền, tượng, các bức chạm nổi...\nGiữa người Chăm với các cư dân Việt ở Nhật Nam, Cửu Chân và Giao Chỉ có mối quan hệ chặt chẽ từ lâu đời. Nhiều cuộc nổi dậy của nhân dân Tường Lâm và Nhật Nam được nhân dân Giao Châu ủng hộ. Nhân dân Tường Lâm và Nhật Nam cũng nổi dậy ứng ứng cuộc khởi nghĩa Hai Bà Trưng.",            
        },
    ],
    # query 4
    [
        {
            "title": "Nhà Đinh xây dựng đất nước",
            "content": "Bài: NƯỚC ĐẠI CỒ VIỆT THỜI ĐINH - TIỀN LÊ - P1: TÌNH HÌNH CHÍNH TRỊ,QUÂN SỰ\nNhà Đinh xây dựng đất nước: Năm 968, công cuộc thống nhất đất nước đã hoàn thành, Đinh Bộ Lĩnh lên ngôi Hoàng đế (Đinh Tiên Hoàng), đặt tên nước là Đại Cồ Việt (nước Việt lớn), đóng đô tại Hoa Lư. Mùa xuân năm 970, vua Đinh đặt niên hiệu là Thái Bình, sai sứ sang giao hảo với nhà Tống.\nĐinh Bộ Lĩnh phong vương cho các con, cử các tướng lĩnh thân cận như Đinh Điền, Nguyễn Bặc, Phạm Hạp, Lê Hoàn.. nắm giữ các chức vụ chủ chốt. Ông cho xây dựng cung điện, đúc tiền để tiêu dùng trong nước; đối với những kẻ phạm tội, thì dùng những hình phạt khắc nghiệt như ném vào vạc đầu sôi, hay vứt vào chuồng hổ.",
        },
    ],
    # query 5
    [
        {
            "title": "Đời sống văn hóa",
            "content": "Bài: NƯỚC ĐẠI CỒ VIỆT THỜI ĐINH - TIỀN LÊ - P2: SỰ PHÁT TRIỂN KINH TẾ VÀ VĂN HÓA\nĐời sống văn hóa: Trong xã hội, vua và các quan văn, võ (cùng một số nhà sư) tạo thành bộ máy thống trị. Những người bị thống trị gồm nông dân, thợ thủ công, người làm nghề buôn bán nhỏ và một số ít địa chủ. Đa số nông dân là những người dân tự do, cày ruộng công làng xã, có quyền lợi gắn bó với làng, với nước. Nô tì, số lượng không nhiều, là tầng lớp dưới cùng của xã hội. Cuộc sống của nhân dân còn đơn giản, bình dị. Giáo dục chùa phát triển. Nho học đã xâm nhập vào nước ta, nhưng chưa tạo được ảnh hưởng đáng kể. Đã có một số nhà sư mở các lớp học ở trong chùa. Đạo Phật được truyền bá rộng rãi. Các nhà sư thường là người có học, giỏi chữ Hán, được nhà nước và nhân dân quý trọng. Những đại sư như Ngô Chân Lưu, Đỗ Thuận, Vạn Hạnh được trọng dụng như những cố vấn cung đình, những nhà ngoại giao đắc lực của nhà vua, nhất là trong các dịp đón tiếp các sứ thần nhà Tống. Chùa chiền được xây dựng ở nhiều nơi. Tại kinh đô Hoa Lư có các chùa Bà Ngô, chùa Tháp, chùa Nhất Trụ...\nNhiều loại hình văn hóa dân gian đã tồn tại trong thời Đinh - Tiền Lê như ca hát, nhảy múa, đua thuyền, đánh đu, đấu võ, đánh vật...",
        },
    ],
    # query 6
    [
        {
            "title": "opening: Nhật bản sau chiến tranh thế giới thứ 2",
            "content": "Bài: Nhật Bản\nOpening: Là nước bại trận trong Chiến tranh thế giới thứ hai, nhưng từ sau năm 1945, Nhật Bản bước vào một thời kì phát triển mới với những đổi thay căn bản về chính trị - xã hội cùng những thành tựu như một sự \"thần kì\" về kinh tế, khoa học - công nghệ. Nhật Bản đã vươn lên, trở thành một siêu cường kinh tế, một trung tâm kinh tế - tài chính thế giới."
        },
        {
            "title": "Nhật Bản từ năm 1945 đến năm 1952",
            "content": "Bài: Nhật Bản\nNhật Bản từ năm 1945 đến năm 1952: Sự thất bại trong Chiến tranh thế giới thứ hai đã để lại cho Nhật Bản những hậu quả hết sức nặng nề.\nKhoảng 3 triệu người chết và mất tích ; 40% đô thị, 80% tàu bè, 34% máy móc công nghiệp bị phá hủy ; 13 triệu người thất nghiệp ; thảm họa đói, rét đe dọa toàn nước Nhật.\nSau chiến tranh, Nhật Bản đã bị quân đội Mĩ, với danh nghĩa lực lượng Đồng minh, chiếm đóng từ năm 1945 đến năm 1952, nhưng Chính phủ Nhật Bản vẫn được phép tồn tại và hoạt động.\nVề chính trị, Bộ Chỉ huy tối cao lực lượng Đồng minh (viết tắt theo tiếng Anh là SCAP) đã loại bỏ chủ nghĩa quân phiệt và bộ máy chiến tranh của Nhật Bản. Tòa án Quân sự Viễn Đông đã xét xử tội phạm chiến tranh Nhật Bản (kết án tử hình 7 tên, tù chung thân 16 tên), Hiến pháp mới do SCAP tổ chức soạn thảo (có hiệu lực từ ngày 3 - 5 - 1947), quy định Nhật Bản là nước quân chủ lập hiến, nhưng thực chất là theo chế độ dân chủ đại nghị tư sản. Hiến pháp mới vẫn duy trì ngôi vị Thiên hoàng song chỉ mang tính tượng trưng, không còn quyền lực đối với Nhà nước ; xác định Nghị viện gồm hai viện, do nhân dân bầu ra, là cơ quan quyền lực tối cao giữ quyền lập pháp ; Chính phủ nắm quyền hành pháp, do Thủ tướng đứng đầu. Nhật Bản cam kết từ bỏ việc tiến hành chiến tranh, không đe dọa hoặc sử dụng vũ lực trong quan hệ quốc tế ; không duy trì quân đội thường trực, chỉ có lực lượng phòng vệ dân sự bảo đảm an ninh, trật tự trong nước.\nVề kinh tế, SCAP đã thực hiện ba cuộc cải cách lớn: một là, thủ tiêu chế độ tập trung kinh tế, trước hết là giải tán các \"Daibátxư\" (tức là các tập đoàn, công ti tư bản lũng đoạn còn mang nhiều tính chất dòng tộc) ; hai là, cải cách ruộng đất, quy định địa chủ chỉ được sở hữu không quá 3 hécta ruộng, số còn lại Chính phủ đem bán cho nông dân ; ba là, dân chủ hóa lao động (thông qua việc thực hiện các đạo luật về lao động). Dựa vào sự nỗ lực của bản thân và viện trợ của Mĩ, đến khoảng năm 1950 - 1951, Nhật Bản đã khôi phục kinh tế, đạt mức trước chiến tranh.\nTrong chính sách đối ngoại, Nhật Bản chủ trương liên minh chặt chẽ với Mĩ. Nhờ đó, nước Nhật sớm kí kết được Hiệp ước hòa bình Xan Phranxixcô (8 - 9 - 1951), chấm dứt chế độ chiếm đóng của Đồng minh (1952). Cùng ngày, Hiệp ước an ninh Mĩ - Nhật được kí kết, đặt nền tảng mới cho quan hệ giữa hai nước. Theo đó, Nhật Bản chấp nhận đứng dưới \"chiếc ô\" bảo hộ hạt nhân của Mĩ, để cho Mĩ đóng quân và xây dựng căn cứ quân sự trên lãnh thổ Nhật Bản.",
        },
        {
            "title": "Nhật Bản từ năm 1952 đến năm 1973",
            "content": "Bài: Nhật Bản\nNhật Bản từ năm 1952 đến năm 1973: Sau khi được phục hồi, từ năm 1952 đến năm 1960, kinh tế Nhật Bản có bước phát triển nhanh, nhất là từ năm 1960 đến năm 1973, thường được gọi là giai đoạn phát triển \"thần kì\".\nTốc độ tăng trưởng bình quân hằng năm của Nhật Bản từ năm 1960 đến năm 1969 là 10,8% ; từ năm 1970 đến năm 1973, tuy có giảm đi nhưng vẫn đạt bình quân bình quân 7,8%, cao hơn rất nhiều so với các nước phát triển khác. Năm 1968, kinh tế Nhật Bản đã vượt Anh, Pháp, Cộng hòa Liên bang Đức, Italia và Canada, vươn lên đứng thứ hai trong thế giới tư bản (sau Mĩ).\nTừ đầu những năm 70 trở đi, Nhật Bản trở thành một trong ba trung tâm kinh tế - tài chính lớn của thế giới (cùng với Mĩ và Tây Âu).\nNhật Bản rất coi trọng giáo dục và khoa học - kĩ thuật, luôn tìm cách đẩy nhanh sự phát triển bằng cách mua bằng phát minh sáng chế. Tính đến năm 1968, Nhật Bản đã mua bằng phát minh của nước ngoài trị giá tới 6 tỉ USD. Khoa học - kĩ thuật và công nghệ Nhật Bản chủ yếu tập trung vào lĩnh vực sản xuất ứng dụng dân dụng, đạt được nhiều thành tựu lớn.\nNgoài các sản phẩm dân dụng nổi tiếng thế giới (như tivi, tủ lạnh, ôtô v.v.), Nhật Bản còn đóng tàu chở dầu có trọng tải 1 triệu tấn ; xây dựng các công trình thế kỉ như đường ngầm dưới biển dài 53,8 km nối hai đảo Hônsu và Hốccaiđô, cầu đường bộ dài 9,4 km nối hai đảo Hônsu và Sicôcư.\nNhật Bản nhanh chóng vươn lên thành một siêu cường kinh tế (sau Mĩ) là do một số yếu tố sau: 1. Ở Nhật Bản, con người được coi là vốn quý nhất, là nhân tố quyết định hàng đầu ; 2. Vai trò lãnh đạo, quản lí có hiệu quả của Nhà nước ; 3. Các công ti Nhật Bản năng động, cổ tầm nhìn xa, quản lí tốt nên có tiềm lực và sức cạnh tranh cao ; 4. Nhật Bản biết áp dụng các thành tựu khoa học - kĩ thuật hiện đại để nâng cao năng suất, chất lượng, hạ giá thành sản phẩm ; 5. Chi phí cho quốc phòng của Nhật Bản thấp (không vượt quá 1% GDP), nên có điều kiện tập trung vốn đầu tư cho kinh tế ; 6. Nhật Bản đã tận dụng tốt các yếu tố bên ngoài để phát triển, như nguồn viện trợ của Mĩ, các cuộc chiến tranh ở Triều Tiên (1950 - 1953) và Việt Nam (1954 - 1975) để làm giàu v.v..\nTuy nhiên, nền kinh tế Nhật Bản vẫn có những hạn chế và gặp phải nhiều khó khăn: 1. Lãnh thổ Nhật Bản không rộng, tài nguyên khoáng sản rất nghèo nàn, nền công nghiệp của Nhật Bản hầu như phụ thuộc vào các nguồn nguyên, nhiên liệu nhập khẩu từ bên ngoài ; 2. Cơ cấu vùng kinh tế của Nhật Bản thiếu cân đối, tập trung chủ yếu vào ba trung tâm là Tôkiô, Ôxaca và Nagôia, giữa công nghiệp và nông nghiệp cũng có sự mất cân đối ; 3. Nhật Bản luôn gặp sự cạnh tranh quyết liệt của Mĩ, Tây Âu, các nước công nghiệp mới, Trung Quốc v.v.\nVề chính trị, từ năm 1955 đến năm 1993, Đảng Dân chủ Tự do (LDP) liên tục cầm quyền ở Nhật Bản. Dưới thời Thủ tướng Ikêđa Hayato (1960 - 1964), Nhật Bản chủ trương xây dựng \"Nhà nước phúc lợi chung\", tăng thu nhập quốc dân lên gấp đôi trong vòng 10 năm (1960 - 1970).\nNền tảng căn bản trong chính sách đối ngoại của Nhật Bản vẫn là liên minh chặt chẽ với Mĩ. Hiệp ước an ninh Mĩ - Nhật (kí năm 1951) có giá trị trong 10 năm, sau đó được kéo dài vĩnh viễn. Tuy vậy, phong trào đấu tranh của nhân dân Nhật Bản chống Hiệp ước an ninh Mĩ - Nhật, chống chiến tranh của Mĩ ở Việt Nam, cũng như các cuộc đấu tranh theo mùa (mùa xuân và mùa thu) kể từ năm 1954 trở đi đòi tăng lương, cải thiện đời sống luôn diễn ra mạnh mẽ.\nNăm 1956, Nhật Bản bình thường hóa quan hệ ngoại giao với Liên Xô. Cùng năm đó, Nhật Bản là thành viên của Liên hợp quốc.",
        },
        {
            "title": "Nhật Bản từ năm 1973 đến năm 1991",
            "content": "Bài: Nhật Bản\nNhật Bản từ năm 1973 đến năm 1991: Do tác động của cuộc khủng hoảng năng lượng thế giới, từ năm 1973 trở đi, sự phát triển kinh tế của Nhật Bản thường xen kẽ với những giai đoạn suy thoái ngắn. Tuy nhiên, từ nửa sau những năm 80, Nhật Bản đã vươn lên thành siêu cường tài chính số một thế giới với lượng dự trữ vàng và ngoại tệ gấp 3 lần của Mĩ, gấp 1,5 lần của Cộng hòa Liên bang Đức. Nhật Bản cũng là chủ nợ lớn nhất thế giới.\nVới tiềm lực kinh tế - tài chính ngày càng lớn mạnh, từ nửa sau những năm 770, Nhật Bản bắt đầu đưa ra chính sách đối ngoại mới, thể hiện trong học thuyết Phucưđa (1977) và học thuyết Kaiphu (1991). Nội dung chủ yếu của các học thuyết trên là tăng cường quan hệ kinh tế, chính trị, văn hóa, xã hội với các nước Đông Nam Á và tổ chức ASEAN.\nNhật Bản thiết lập quan hệ ngoại giao với Việt Nam ngày 21 - 9 - 1973.",
        },
        {
            "title": "Nhật Bản từ năm 1991 đến năm 2000",
            "content": "Bài: Nhật Bản\nNhật Bản từ năm 1991 đến năm 2000: Từ đầu thập kỉ 90, kinh tế Nhật Bản lâm vào tình trạng suy thoái, nhưng Nhật Bản vẫn là một trong ba trung tâm kinh tế - tài chính lớn của thế giới.\nTỉ trọng của Nhật Bản trong nền sản xuất của thế giới là 1/10. GDP của Nhật Bản năm 2000 là 4746 tỉ USD và bình quân GDP trên đầu người là 37408 USD.\nKhoa học - kĩ thuật của Nhật Bản vẫn tiếp tục phát triển ở trình độ cao. Tính đến năm 1992, Nhật Bản đã phóng 49 vệ tinh khác nhau và hợp tác có hiệu quả với Mĩ, Liên Xô (sau là Liên bang Nga), trong các chương trình vũ trụ quốc tế.\nVề văn hóa, tuy là một nước tư bản phát triển cao, nhưng Nhật Bản vẫn giữ được những giá trị truyền thống và bản sắc văn hóa của mình. Sự kết hợp hài hòa giữa truyền thống và hiện đại là nét đáng chú ý trong đời sống văn hóa Nhật Bản.\nVề chính trị, sau 38 năm Đảng Dân chủ Tự do liên tục cầm quyền (1955 - 1993), từ năm 1993 đến năm 2000, chính quyền ở Nhật Bản thuộc về các đảng đối lập hoặc liên minh các đảng phái khác nhau, tình hình xã hội Nhật Bản có phần không ổn định.\nTrận động đất ở Côbê (1 - 1995) đã gây thiệt hại lớn về người và của ; vụ khủng bố bằng hơi độc trong đường tàu điện ngầm của giáo phái Aum (3 - 1995) và nạn thất nghiệp tăng cao v.v. đã làm cho nhiều người dân Nhật Bản hết sức lo lắng.\nVề đối ngoại, Nhật Bản tiếp tục duy trì sự liên minh chặt chẽ với Mĩ. Tháng 4 - 1996 hai nước ra tuyên bố khẳng định lại việc kéo dài vĩnh viễn Hiệp ước an ninh Mĩ - Nhật Mặt khác, với học thuyết Miyadaoa (1 - 1993), và học thuyết Hasimôtô (1 - 1997), Nhật Bản vẫn coi trọng quan hệ với Tây Âu, mở rộng hoạt động đối ngoại với các đối tác khác đến phạm vi toàn cầu và chú trọng phát triển quan hệ với các nước Đông Nam Á.\nTừ đầu những năm 90, Nhật Bản nỗ lực vươn lên thành một cường quốc chính trị để tương xứng với vị thế siêu cường kinh tế.",
        },
    ],
    # query 7
    [
        {
            "title": "Cách mạng khoa học - công nghệ và xu thế toàn cầu hóa",
            "content": "Chương: Cách mạng khoa học - công nghệ và xu thế toàn cầu hóa",
        },
        {
            "title": "Cách mạng khoa học - công nghệ và xu thế toàn cầu hóa nửa sau thế kỉ XX",
            "content": "Chương: Cách mạng khoa học - công nghệ và xu thế toàn cầu hóa\nBài: Cách mạng khoa học - công nghệ và xu thế toàn cầu hóa nửa sau thế kỉ XX",
        },
        {
            "title": "Opening",
            "content": "Bài: Cách mạng khoa học - công nghệ và xu thế toàn cầu hóa nửa sau thế kỉ XX\nOpening: Từ những năm 40 của thế kỉ XX, trên thế giới đã diễn ra cuộc cách mạng khoa học - kĩ thuật hiện đại, khởi đầu từ nước Mĩ. Với quy mô rộng lớn, nội dung sâu sắc và toàn diện, nhịp điệu vô cùng nhanh chóng, cuộc cách mạng khoa học - kĩ thuật đã đưa lại biết bao thành tựu kì diệu và những đổi thay to lớn trong đời sống nhân loại. Nền văn minh thế giới có những bước nhảy vọt mới.",
        },
        {
            "title": "Nguồn gốc và đặc điểm của cuộc cách mạng khoa học - công nghệ",
            "content": "Bài: Cách mạng khoa học - công nghệ và xu thế toàn cầu hóa nửa sau thế kỉ XX\nNguồn gốc và đặc điểm của cuộc cách mạng khoa học - công nghệ: Cũng như cách mạng công nghiệp ở thế kỉ XVIII - XIX, cuộc cách mạng khoa cuộc cách mạng khoa học - kĩ thuật ngày nay diễn ra là do những đòi hỏi của cuộc sống, của sản xuất nhằm đáp ứng nhu cầu vật chất và tinh thần ngày càng cao của con người, nhất là trong tình hình bùng nổ dân số thế giới và sự vơi cạn nghiêm trọng các nguồn tài nguyên thiên nhiên, đặc biệt từ sau Chiến tranh thế giới thứ hai.\nĐặc điểm lớn nhất của cách mạng khoa học - kĩ thuật ngày nay là khoa học trở thành lực lượng sản xuất trực tiếp. Khác với cách mạng công nghiệp thế kỉ XVIII, trong cuộc cách mạng khoa học - kĩ thuật hiện đại, mọi phát minh kĩ thuật đều bắt nguồn từ nghiên cứu khoa học. Khoa học gắn liền với kĩ thuật, khoa học đi trước mở đường cho kĩ thuật. Đến lượt mình, kĩ thuật lại đi trước mở đường cho sản xuất. Khoa học đã tham gia trực tiếp vào sản xuất, đã trở thành nguồn gốc chính của những tiến bộ kĩ thuật và công nghệ.\nCuộc cách mạng khoa học - kĩ thuật ngày nay đã phát triển qua hai giai đoạn: giai đoạn đầu từ những năm 40 đến nửa đầu những năm 70 của thế kỉ XX ; giai đoạn thứ hai từ sau cuộc khủng hoảng năng lượng năm 1973 đến nay. Trong giai đoạn sau, cuộc cách mạng chủ yếu diễn ra về công nghệ với sự ra đời của thế hệ máy tính điện tử mới (thế hệ thứ ba), về vật liệu mới, về những dạng năng lượng mới và công nghệ sinh học, phát triển tin học. Cuộc cách mạng công nghệ trở thành cốt lõi của cách mạng khoa học - kĩ thuật nên giai đoạn thứ hai đã được gọi là cách mạng khoa học - công nghệ.",
        },
        {
            "title": "Những thành tựu tiêu biểu của cuộc cách mạng khoa học - công nghệ",
            "content": "Bài: Cách mạng khoa học - công nghệ và xu thế toàn cầu hóa nửa sau thế kỉ XX\nNhững thành tựu tiêu biểu của cuộc cách mạng khoa học - công nghệ: Trải qua hơn nửa thế kỉ, nhất là từ sau những năm 70, cuộc cách mạng khoa học - kĩ thuật đã thu được những tiến bộ phi thường và những thành tựu kì diệu.\nTrong lĩnh vực khoa học cơ bản, loài người đã đạt được những thành tựu hết sức to lớn, những bước nhảy vọt chưa từng thấy trong lịch sử các ngành Toán học, Vật lí học, Hóa học, Sinh học v.v.. Dựa vào những phát minh lớn của các ngành khoa học cơ bản, con người đã ứng dụng cải tiến kĩ thuật, phục vụ sản xuất và cuộc sống của mình.\nSự kiện gây chấn động lớn trong dư luận thế giới là tháng 3 - 1997, các nhà khoa học đã tạo ra được con cừu Đôli bằng phương pháp sinh sản vô tính từ một tế bào lấy từ tuyến vú của một con cừu đang có thai. Tháng 6 - 2000 sau 10 năm hợp tác nghiên cứu, các nhà khoa học của các nước Anh, Pháp, Mĩ, Đức, Nhật Bản và Trung Quốc đã công bố \"Bản đồ gen người\". Đến tháng 4 - 2003, \"Bản đồ gen người\" mới được giải mã hoàn chỉnh.\nNhững thành tựu này đã mở ra một kỉ nguyên mới của Y học và Sinh học, với những triển vọng to lớn, đẩy lùi bệnh tật và tuổi già. Tuy nhiên, những thành tựu này lại gây nên những lo ngại về mặt pháp lí và đạo lí như công nghệ sao chép con người hoặc thương mại hóa công nghệ gen Trong lĩnh vực công nghệ, đã xuất hiện những phát minh quan trọng, đạt được những thành tựu to lớn: những công cụ sản xuất mới (máy tính điện tử, máy tự động và hệ thống máy tự động, rôbốt v.v.) ; những nguồn năng lượng mới (năng lượng mặt trời, năng lượng gió và nhất là năng lượng nguyên tử v.v.) ; những vật liệu mới (như chất pôlime - chất dẻo với nhiều loại hình khác nhau, các loại vật liệu siêu sạch, siêu cứng, siêu bền, siêu dẫn...) ; công nghệ sinh học với những đột phá phi thường trong công nghệ di truyền, công nghệ tế bào, công nghệ vi sinh và công nghệ enzim, ... dẫn tới cuộc \"cách mạng xanh\" trong nông nghiệp với những giống lúa mới có năng suất cao, chịu bệnh tốt ; những tiến bộ thần kì trong thông tin liên lạc và giao thông vận tải (cáp sợi thủy tinh quang dẫn, máy bay siêu âm khổng lồ, tàu hỏa tốc độ cao v.v.) ; chinh phục vũ trụ (vệ tinh nhân tạo, du hành vũ trụ v.v.).\nTrong những thập niên gần đây, công nghệ thông tin đã phát triển mạnh mẽ như một sự bùng nổ trên phạm vi toàn cầu. Hiện nay, máy tính, đặc biệt là máy vi tính, đang được sử dụng ở khắp mọi nơi và có khả năng liên kết với nhau bởi các mạng truyền dữ liệu, hình thành mạng thông tin máy tính toàn cầu (Internet). Công nghệ thông tin ngày càng được ứng dụng sâu rộng trong mọi ngành kinh tế và hoạt động xã hội. Có thể nói, ngày nay nền văn minh nhân loại đã sang một chương mới - \"văn minh thông tin\".\nCuộc cách mạng khoa học - công nghệ có những tác động tích cực về nhiều mặt như tăng năng suất lao động, không ngừng nâng cao mức sống và chất lượng cuộc sống của con người. Từ đó dẫn đến những thay đổi lớn về cơ cấu dân cư, chất lượng nguồn nhân lực, những đòi hỏi mới về giáo dục và đào tạo nghề nghiệp, sự hình thành một thị trường thế giới với xu thế toàn cầu hóa.\nTuy nhiên, cuộc cách mạng khoa học - công nghệ cũng gây nên những hậu quả tiêu cực (chủ yếu do chính con người tạo nên) như tình trạng ô nhiễm môi trường trên hành tinh cũng như trong vũ trụ, hiện tượng Trái Đất nóng dần lên, những tai nạn lao động và giao thông, các loại dịch bệnh mới v.v. và nhất là việc chế tạo những loại vũ khí hiện đại có sức công phá và hủy diệt khủng khiếp, có thể tiêu diệt nhiều lần sự sống trên hành tinh.",
        },
        {
            "title": "Xu thế toàn cầu hóa và ảnh hưởng của nó",
            "content": "Bài: Cách mạng khoa học - công nghệ và xu thế toàn cầu hóa nửa sau thế kỉ XX\nXu thế toàn cầu hóa và ảnh hưởng của nó: Một hệ quả quan trọng của cách mạng khoa học - công nghệ là từ đầu những năm 80 của thế kỉ XX, nhất là từ sau Chiến tranh lạnh, trên thế giới đã diễn ra xu thế toàn cầu hóa.\nXét về bản chất, toàn cầu hóa là quá trình tăng lên mạnh mẽ những mối liên hệ.\nNhững ảnh hưởng tác động lẫn nhau, phụ thuộc lẫn nhau của tất cả các khu vực, các quốc gia, các dân tộc trên thế giới.\nNhững biểu hiện chủ yếu của xu thế toàn cầu hóa ngày nay là: \n- Sự phát triển nhanh chóng của quan hệ thương mại quốc tế. Từ sau Chiến tranh thế giới thứ hai đến cuối thập kỉ 90, giá trị trao đổi thương mại trên phạm vi quốc tế đã tăng 12 lần. Thương mại quốc tế tăng có nghĩa là nền kinh tế của các nước trên thế giới có quan hệ chặt chẽ và phụ thuộc lẫn nhau, tính quốc tế hóa của nền kinh tế thế giới tăng.\n- Sự phát triển và tác động to lớn của các công ti xuyên quốc gia. Theo số liệu của Liên hợp quốc, khoảng 500 công ti xuyên quốc gia lớn kiểm soát tới 25% tổng sản phẩm thế giới và giá trị trao đổi của những công ti này tương đương 3/4 giá trị thương mại toàn cầu.\n- Sự sáp nhập và hợp nhất các công ti thành những tập đoàn lớn, nhất là các công ti khoa học - kĩ thuật, nhằm tăng cường khả năng cạnh tranh trên thị trường trong và ngoài nước. Làn sóng sáp nhập này tăng lên nhanh chóng vào những năm cuối thế kỉ XX.\n- Sự ra đời của các tổ chức liên kết kinh tế, thương mại, tài chính quốc tế và khu vực. Đó là Quỹ Tiền tệ Quốc tế (IMF), Ngân hàng Thế giới (WB), Tổ chức Thương mại Thế giới (WTO), Liên minh châu Âu (EU), Hiệp ước Thương mại tự do Bắc Mĩ (NAFTA), Khu vực Thương mại tự do ASEAN (AFTA), Diễn đàn hợp tác kinh tế châu Á - Thái Bình Dương (APEC), Diễn đàn hợp tác Á - Âu tác kinh tế châu (ASEM) v.v.. Các tổ chức này có vai trò ngày càng quan trọng trong việc giải quyết những vấn đề kinh tế chung của thế giới và khu vực.\nLà kết quả của quá trình tăng tiến mạnh mẽ của lực lượng sản xuất, toàn cầu hóa là xu thế khách quan, là một thực tế không thể đảo ngược được. Nó có mặt tích cực và mặt tiêu cực, nhất là đối với các nước đang phát triển.\nVề mặt tích cực, đó là thúc đẩy rất mạnh, rất nhanh sự phát triển và xã hội hóa của lực lượng sản xuất, đưa lại sự tăng trưởng cao (nửa đầu thế kỉ XX, GDP thế giới tăng 2,7 lần, nửa cuối thế kỉ tăng 5,2 lần). góp phần chuyển biến cơ cấu kinh tế, đòi hỏi phải tiến hành cải cách sâu rộng để nâng cao sức cạnh tranh và hiệu quả của nền kinh tế.\nVề mặt tiêu cực, toàn cầu hóa làm trầm trọng thêm sự bất công xã hội, đào sâu hố ngăn cách giàu - nghèo trong từng nước và giữa các nước. Toàn cầu hóa làm cho mọi mặt hoạt động và đời sống của con người kém an toàn (từ kém an toàn về kinh tế, tài chính đến kém an toàn về chính trị), hoặc tạo ra nguy cơ đánh mất bản sắc dân tộc và xâm phạm nền độc lập tự chủ của các quốc gia v.v..\nNhư thế, toàn cầu hóa là thời cơ lịch sử, là cơ hội rất to lớn cho các nước phát triển mạnh mẽ, đồng thời cũng tạo ra những thách thức to lớn. Việt Nam cũng nằm trong xu thế chung đó. Do vậy, \"Nắm bắt cơ hội, vượt qua thách thức, phát triển mạnh mẽ trong thời kì mới, đó là vấn đề có ý nghĩa sống còn đối với Đảng và nhân dân ta\".",
        },
    ],
    # query 8

    # query 9

    # query 10

]

# Xử lý dataset

In [31]:
# Bỏ qua "chapter"
# Bỏ qua "lesson"
# Đối với "title", sẽ được bổ sung thêm nội dung bằng cách chèn thêm chapter và lesson. 
# => Nhằm tăng độ chính xác khả năng tìm kiếm

df["embedding_content"] = None
df["tokenization"] = None
df["num_of_tokens"] = None
df["embeded"] = None

chapter = ""
lesson = ""
addition_content = ""

for idx in range(len(df)):
    if df.loc[idx, "type"] == "chapter":
        chapter = df.loc[idx, "title"]
    elif df.loc[idx, "type"] == "lesson":
        lesson = df.loc[idx, "title"]
    elif df.loc[idx, "type"] == "title":
        addition_content = chapter + " " + lesson
        df.loc[idx, "embedding_content"] = addition_content + " " + df.loc[idx, "content"]
    else:
        print(f"Có một đoạn data bị sai, tại dòng {idx}")
        break
df.head()

,title,content,type,embedding_content,tokenization,num_of_tokens,embeded
0,BUỔI ĐẦU LỊCH SỬ NƯỚC TA,Chương: BUỔI ĐẦU LỊCH SỬ NƯỚC TA,chapter,None,None,None,None
1,THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA,Chương: BUỔI ĐẦU LỊCH SỬ NƯỚC TA\nBài: THỜI NG...,lesson,None,None,None,None
2,Những dấu tích của Người tối cổ được tìm thấy ...,Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA\nNhững ...,title,BUỔI ĐẦU LỊCH SỬ NƯỚC TA THỜI NGUYÊN THUỶ TRÊN...,None,None,None
3,"Ở giai đoạn đầu, Người tinh khôn sống như thế ...",Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA\nỞ giai...,title,BUỔI ĐẦU LỊCH SỬ NƯỚC TA THỜI NGUYÊN THUỶ TRÊN...,None,None,None
4,Giai đoạn phát triển của Người tinh khôn có gì...,Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA\nGiai đ...,title,BUỔI ĐẦU LỊCH SỬ NƯỚC TA THỜI NGUYÊN THUỶ TRÊN...,None,None,None


In [20]:
for idx in range(5):
    print("idx = {idx}".format(idx=idx))
    print(df.loc[idx, "embedding_content"])

idx = 0
None
idx = 1
None
idx = 2
BUỔI ĐẦU LỊCH SỬ NƯỚC TA THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA
Những dấu tích của Người tối cổ được tìm thấy ở đâu?: Thời xa xưa, nước ta là một vùng rừng núi rậm rạp với nhiều hang động, mái đá, nhiều sông suối, có vùng ven biển dài; khí hậu hai mùa nóng - lạnh rõ rệt, thuận lợi cho cuộc sống của cỏ cây, muông thú và con người. Vào những năm 1960 - 1965, các nhà khảo cổ học đã lần lượt phát hiện được hàng loạt di tích của Người tối cổ. Ở các hang Thẩm Khuyên, Thẩm Hai (Lạng Sơn), trong lớp đất chứa nhiều than, xương vật cổ cách đây 40 - 30 vạn năm, người ta phát hiện được những chiếc răng của Người tối cổ. Ở một số nơi khác như núi Đọ, Quan Yên (Thanh Hoá), Xuân Lộc (Đồng Nai), người ta phát hiện được nhiều công cụ đá thô sơ dùng để chặt, đập; nhiều mảnh đá ghè mỏng. Ở nhiều chỗ.
idx = 3
BUỔI ĐẦU LỊCH SỬ NƯỚC TA THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA
Ở giai đoạn đầu, Người tinh khôn 

In [49]:
# Độ dài câu dài nhất
print(f"Độ dài câu dài nhất là: {max([len(df.loc[idx, "embedding_content"]) for idx in range(len(df)) if df.loc[idx, "embedding_content"] is not None])}")
print(f"Độ dài câu ngắn nhất là: {min([len(df.loc[idx, "embedding_content"]) for idx in range(len(df)) if df.loc[idx, "embedding_content"] is not None])}")

Độ dài câu dài nhất là: 6992
Độ dài câu ngắn nhất là: 176


# vietnamese-bi-encoder

## Load model

In [ ]:
embedding_model_path = "../../embedding_models/vietnamese-bi-encoder"
tokenizer = AutoTokenizer.from_pretrained(embedding_model_path)
model = SentenceTransformer(embedding_model_path)

In [ ]:
print(f"tokenizer model max length: {tokenizer.model_max_length}")
print(f"embedding model dimension: {model.get_sentence_embedding_dimension()}")

tokenizer model max length: 512
embedding model dimension: 384


## Xử lý dữ liệu

In [ ]:
df_1 = copy.deepcopy(df)
df_1.head()

,title,content,type,embedding_content,tokenization,num_of_tokens,embeded
0,BUỔI ĐẦU LỊCH SỬ NƯỚC TA,Chương: BUỔI ĐẦU LỊCH SỬ NƯỚC TA,chapter,None,None,None,None
1,THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA,Chương: BUỔI ĐẦU LỊCH SỬ NƯỚC TA\nBài: THỜI NG...,lesson,None,None,None,None
2,Những dấu tích của Người tối cổ được tìm thấy ...,Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA\nNhững ...,title,BUỔI ĐẦU LỊCH SỬ NƯỚC TA THỜI NGUYÊN THUỶ TRÊN...,None,None,None
3,"Ở giai đoạn đầu, Người tinh khôn sống như thế ...",Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA\nỞ giai...,title,BUỔI ĐẦU LỊCH SỬ NƯỚC TA THỜI NGUYÊN THUỶ TRÊN...,None,None,None
4,Giai đoạn phát triển của Người tinh khôn có gì...,Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA\nGiai đ...,title,BUỔI ĐẦU LỊCH SỬ NƯỚC TA THỜI NGUYÊN THUỶ TRÊN...,None,None,None


In [ ]:
df_1 = df_1.dropna(subset=["embedding_content"]).reset_index(drop=True)
print(len(df_1))
df_1.head()

855


,title,content,type,embedding_content,tokenization,num_of_tokens,embeded
0,Những dấu tích của Người tối cổ được tìm thấy ...,Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA\nNhững ...,title,BUỔI ĐẦU LỊCH SỬ NƯỚC TA THỜI NGUYÊN THUỶ TRÊN...,None,None,None
1,"Ở giai đoạn đầu, Người tinh khôn sống như thế ...",Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA\nỞ giai...,title,BUỔI ĐẦU LỊCH SỬ NƯỚC TA THỜI NGUYÊN THUỶ TRÊN...,None,None,None
2,Giai đoạn phát triển của Người tinh khôn có gì...,Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA\nGiai đ...,title,BUỔI ĐẦU LỊCH SỬ NƯỚC TA THỜI NGUYÊN THUỶ TRÊN...,None,None,None
3,Đời sống vật chất,Bài: ĐỜI SỐNG CỦA NGƯỜI NGUYÊN THỦY TRÊN ĐẤT N...,title,BUỔI ĐẦU LỊCH SỬ NƯỚC TA ĐỜI SỐNG CỦA NGƯỜI NG...,None,None,None
4,Tổ chức xã hội,Bài: ĐỜI SỐNG CỦA NGƯỜI NGUYÊN THỦY TRÊN ĐẤT N...,title,BUỔI ĐẦU LỊCH SỬ NƯỚC TA ĐỜI SỐNG CỦA NGƯỜI NG...,None,None,None


## Tokenization và Embedding

In [ ]:
start = time.time()
for idx in range(len(df_1)):
    df_1.loc[idx, "tokenization"] = tokenizer.encode(df_1.loc[idx, "embedding_content"])
    df_1.loc[idx, "num_of_tokens"] = len(df_1.loc[idx, "tokenization"])
    df_1.loc[idx, "embeded"] = model.encode(df_1.loc[idx, "embedding_content"]).tolist()

total_time = time.time() - start
print(f"Total time: {total_time}")
print(f"Time on a sample: {total_time/len(df_1)}")
df_1.head()

Time: 72.8199770450592


,title,content,type,embedding_content,tokenization,num_of_tokens,embeded
0,Những dấu tích của Người tối cổ được tìm thấy ...,Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA\nNhững ...,title,BUỔI ĐẦU LỊCH SỬ NƯỚC TA THỜI NGUYÊN THUỶ TRÊN...,"[101, 20934, 10448, 1102, 4887, 5622, 2818, 10...",344,"[-0.036770787090063095, -0.01326281949877739, ..."
1,"Ở giai đoạn đầu, Người tinh khôn sống như thế ...",Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA\nỞ giai...,title,BUỔI ĐẦU LỊCH SỬ NƯỚC TA THỜI NGUYÊN THUỶ TRÊN...,"[101, 20934, 10448, 1102, 4887, 5622, 2818, 10...",300,"[-0.07854537665843964, 0.008539889007806778, -..."
2,Giai đoạn phát triển của Người tinh khôn có gì...,Bài: THỜI NGUYÊN THUỶ TRÊN ĐẤT NƯỚC TA\nGiai đ...,title,BUỔI ĐẦU LỊCH SỬ NƯỚC TA THỜI NGUYÊN THUỶ TRÊN...,"[101, 20934, 10448, 1102, 4887, 5622, 2818, 10...",317,"[-0.054575275629758835, 0.01107677910476923, -..."
3,Đời sống vật chất,Bài: ĐỜI SỐNG CỦA NGƯỜI NGUYÊN THỦY TRÊN ĐẤT N...,title,BUỔI ĐẦU LỊCH SỬ NƯỚC TA ĐỜI SỐNG CỦA NGƯỜI NG...,"[101, 20934, 10448, 1102, 4887, 5622, 2818, 10...",365,"[-0.045304831117391586, -0.03193192183971405, ..."
4,Tổ chức xã hội,Bài: ĐỜI SỐNG CỦA NGƯỜI NGUYÊN THỦY TRÊN ĐẤT N...,title,BUỔI ĐẦU LỊCH SỬ NƯỚC TA ĐỜI SỐNG CỦA NGƯỜI NG...,"[101, 20934, 10448, 1102, 4887, 5622, 2818, 10...",286,"[-0.07681848108768463, -0.011844064109027386, ..."


In [ ]:
print(f"Max tokens: {max([df_1.loc[idx, "num_of_tokens"] for idx in range(len(df_1))])}")
print(f"Min tokens: {min([df_1.loc[idx, "num_of_tokens"] for idx in range(len(df_1))])}")

print(f"Length of embeded: {max([len(df_1.loc[idx, "embeded"]) for idx in range(len(df_1))])}")

Max tokens: 2828
Min tokens: 74
Length of embeded: 384


## Khởi tạo qdrant collection

In [ ]:
from qdrant_client import models
from qdrant_client import QdrantClient

client = QdrantClient(url="http://localhost:6333")
# http://localhost:6333/dashboard#/collections

In [ ]:
collection_name = "vietnamese-bi-encoder"
if client.collection_exists(collection_name=collection_name):
    print("Collection already exists.")
else:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=model.get_sentence_embedding_dimension(),
            # distance=models.Distance.DOT
            distance=models.Distance.COSINE
        )
    )

Collection already exists.


## Upload data vào collection

In [ ]:
for idx in range(len(df_1)):
    id_ = idx
    vector = df_1["embeded"][idx]
    payload = {"embedding_content": df_1["embedding_content"][idx],}
    client.upsert(
        collection_name=collection_name,
        points=[models.PointStruct(id=id_, vector=vector, payload=payload)]
    )

## Đánh giá embedding

In [ ]:
# Embedding for queries and ground truth
embeded_queries = model.encode(queries).tolist()

embeded_ground_truths = []
for idx in range(len(ground_truths)):
    embeded_ground_truth = []
    for i in range(len(ground_truths[idx])):
        sentence = ground_truths[idx][i]["title"] + " " + ground_truths[idx][i]["content"]
        embeded_ground_truth.append(model.encode(sentence).tolist())
    embeded_ground_truths.append(embeded_ground_truth)

In [ ]:
def retrieval_embed_query(query, ground_truth, embedding_model, k=5):
    embeded_query = embedding_model.encode(query).tolist()
    results = client.query_points(
        collection_name=collection_name,
        query=embeded_query,
        limit=k,
    ).points

    print(f"Content of query: {query}")
    print(f"Content of ground_truth: {ground_truth}")
    print("---- CÁC KẾT QUẢ RETRIEVAL ----")
    for result in results:
        print(f"Content: {result.payload['embedding_content']}\nscore: {result.score}\n")

In [ ]:
idx = 1
retrieval_embed_query(query=queries[idx], ground_truth=ground_truths[idx], embedding_model=model, k=5)

Content of query: Vào thời Văn Lang - Âu Lạc, công cụ sản xuất được cải tiến như thế nào?
Content of ground_truth: [{'title': 'Công cụ sản xuất được cải tiến như thế nào ?', 'content': 'Bài: NHỮNG CHUYỂN BIẾN TRONG ĐỜI SỐNG KINH TẾ\nCông cụ sản xuất được cải tiến như thế nào ?: Những người nguyên thủy trên đất nước ta tiếp tục mở rộng vùng cư trú. Một số đã dừng lại ở các vùng chân núi, thung lũng ven khe, suối..., một số khác thì chuyển xuống các vùng đất bãi ven sông, dựng chòi, cuộc đất trồng trọt, làm chuồng nuôi lợn, gà, chó... Các nhà khảo cổ đã phát hiện được rất nhiều địa điểm chứa đựng những lưỡi rìu đá có vai được mài rộng ra hai mặt, những lưỡi đục, những bàn mài và những mảnh của đá. Số công cụ bằng xương, sừng cũng nhiều hơn. Bên cạnh đó, họ còn tìm thấy nhiều loại hình đồ gốm như bình, vò, nồi cùng nhiều hạt chuỗi đá, vỏ ốc... Người nguyên thuy cũng đã biết làm chì lưới bằng đất nung để đánh cá.\nTrong một số điểm như Phùng Nguyên (Phú Thọ), Hoa Lộc (Thanh Hoá), Lung Leng

In [ ]:
idx = 2
retrieval_embed_query(query=queries[idx], ground_truth=ground_truths[idx], embedding_model=model, k=5)

Content of query: Hãy kể về lịch sử nước cham-pa?
Content of ground_truth: [{'title': 'NƯỚC CHAM-PA TỪ THẾ KỶ II ĐẾN THẾ KỶ X', 'content': 'Chương: THỜI KÌ BẮC THUỘC VÀ ĐẤU TRANH GIÀNH ĐỘC LẬP\nBài: NƯỚC CHAM-PA TỪ THẾ KỶ II ĐẾN THẾ KỶ X'}, {'title': 'Nước Cham-pa độc lập ra đời', 'content': 'Bài: NƯỚC CHAM-PA TỪ THẾ KỶ II ĐẾN THẾ KỶ X\nNước Cham-pa độc lập ra đời: Thời Hán, sau khi chiếm được Giao Chỉ và Cửu Chân, quân Hán đánh xuống phía nam chiếm cả đất của người Chăm cổ, sáp nhập vào Nhật Nam, đặt ra huyện Tường Lâm.\nQuận Nhật Nam (từ Hoành Sơn trở vào) gồm năm huyện. Huyện xa nhất là Tường Lâm (nay thuộc đất Quảng Nam, Quảng Ngãi, Bình Định), là địa bàn sinh sống của bộ lạc Dừa - tước người Chăm cổ, thuộc nền văn hóa đồng thời Sa Huỳnh khá phát triển.\nVào thế kỉ I, nhân dân Giao Châu nhiều lần nổi dậy. Nhà Hán tới ra bất lực, nhất là đối với các quận xa. Năm 192 - 193, nhân dân Tường Lâm dưới sự lãnh đạo của Khu Liên đã nổi dậy giành độc lập. Khu Liên tự xưng làm vua, đặt tên nư

In [ ]:
idx = 3
retrieval_embed_query(query=queries[idx], ground_truth=ground_truths[idx], embedding_model=model, k=5)

Content of query: Đinh Bộ Lĩnh lên ngôi đặt tên nước là gì?
Content of ground_truth: [{'title': 'Nhà Đinh xây dựng đất nước', 'content': 'Bài: NƯỚC ĐẠI CỒ VIỆT THỜI ĐINH - TIỀN LÊ - P1: TÌNH HÌNH CHÍNH TRỊ,QUÂN SỰ\nNhà Đinh xây dựng đất nước: Năm 968, công cuộc thống nhất đất nước đã hoàn thành, Đinh Bộ Lĩnh lên ngôi Hoàng đế (Đinh Tiên Hoàng), đặt tên nước là Đại Cồ Việt (nước Việt lớn), đóng đô tại Hoa Lư. Mùa xuân năm 970, vua Đinh đặt niên hiệu là Thái Bình, sai sứ sang giao hảo với nhà Tống.\nĐinh Bộ Lĩnh phong vương cho các con, cử các tướng lĩnh thân cận như Đinh Điền, Nguyễn Bặc, Phạm Hạp, Lê Hoàn.. nắm giữ các chức vụ chủ chốt. Ông cho xây dựng cung điện, đúc tiền để tiêu dùng trong nước; đối với những kẻ phạm tội, thì dùng những hình phạt khắc nghiệt như ném vào vạc đầu sôi, hay vứt vào chuồng hổ.'}]
---- CÁC KẾT QUẢ RETRIEVAL ----
Content: Các nước tư bản chủ nghĩa giữa hai cuộc chiến tranh thế giới (1918 - 1939) Nước Đức giữa hai cuộc chiến tranh thế giới (1918 - 1939) Bài:

In [ ]:
idx = 4
retrieval_embed_query(query=queries[idx], ground_truth=ground_truths[idx], embedding_model=model, k=5)

Content of query: Nho giáo bắt đầu hình thành ở nước ta bắt đầu từ khi nào?
Content of ground_truth: [{'title': 'Đời sống văn hóa', 'content': 'Bài: NƯỚC ĐẠI CỒ VIỆT THỜI ĐINH - TIỀN LÊ - P2: SỰ PHÁT TRIỂN KINH TẾ VÀ VĂN HÓA\nĐời sống văn hóa: Trong xã hội, vua và các quan văn, võ (cùng một số nhà sư) tạo thành bộ máy thống trị. Những người bị thống trị gồm nông dân, thợ thủ công, người làm nghề buôn bán nhỏ và một số ít địa chủ. Đa số nông dân là những người dân tự do, cày ruộng công làng xã, có quyền lợi gắn bó với làng, với nước. Nô tì, số lượng không nhiều, là tầng lớp dưới cùng của xã hội. Cuộc sống của nhân dân còn đơn giản, bình dị. Giáo dục chùa phát triển. Nho học đã xâm nhập vào nước ta, nhưng chưa tạo được ảnh hưởng đáng kể. Đã có một số nhà sư mở các lớp học ở trong chùa. Đạo Phật được truyền bá rộng rãi. Các nhà sư thường là người có học, giỏi chữ Hán, được nhà nước và nhân dân quý trọng. Những đại sư như Ngô Chân Lưu, Đỗ Thuận, Vạn Hạnh được trọng dụng như những cố vấn cu

In [ ]:
idx = 5
retrieval_embed_query(query=queries[idx], ground_truth=ground_truths[idx], embedding_model=model, k=5)

Content of query: Lịch sử Nhật Bản sau chiến tranh thế giới thứ 2
Content of ground_truth: [{'title': 'opening: Nhật bản sau chiến tranh thế giới thứ 2', 'content': 'Bài: Nhật Bản\nOpening: Là nước bại trận trong Chiến tranh thế giới thứ hai, nhưng từ sau năm 1945, Nhật Bản bước vào một thời kì phát triển mới với những đổi thay căn bản về chính trị - xã hội cùng những thành tựu như một sự "thần kì" về kinh tế, khoa học - công nghệ. Nhật Bản đã vươn lên, trở thành một siêu cường kinh tế, một trung tâm kinh tế - tài chính thế giới.'}, {'title': 'Nhật Bản từ năm 1945 đến năm 1952', 'content': 'Bài: Nhật Bản\nNhật Bản từ năm 1945 đến năm 1952: Sự thất bại trong Chiến tranh thế giới thứ hai đã để lại cho Nhật Bản những hậu quả hết sức nặng nề.\nKhoảng 3 triệu người chết và mất tích ; 40% đô thị, 80% tàu bè, 34% máy móc công nghiệp bị phá hủy ; 13 triệu người thất nghiệp ; thảm họa đói, rét đe dọa toàn nước Nhật.\nSau chiến tranh, Nhật Bản đã bị quân đội Mĩ, với danh nghĩa lực lượng Đồng min

In [ ]:
idx = 6
retrieval_embed_query(query=queries[idx], ground_truth=ground_truths[idx], embedding_model=model, k=5)

Content of query: Khoa học công nghệ
Content of ground_truth: [{'title': 'Cách mạng khoa học - công nghệ và xu thế toàn cầu hóa', 'content': 'Chương: Cách mạng khoa học - công nghệ và xu thế toàn cầu hóa'}, {'title': 'Cách mạng khoa học - công nghệ và xu thế toàn cầu hóa nửa sau thế kỉ XX', 'content': 'Chương: Cách mạng khoa học - công nghệ và xu thế toàn cầu hóa\nBài: Cách mạng khoa học - công nghệ và xu thế toàn cầu hóa nửa sau thế kỉ XX'}, {'title': 'Opening', 'content': 'Bài: Cách mạng khoa học - công nghệ và xu thế toàn cầu hóa nửa sau thế kỉ XX\nOpening: Từ những năm 40 của thế kỉ XX, trên thế giới đã diễn ra cuộc cách mạng khoa học - kĩ thuật hiện đại, khởi đầu từ nước Mĩ. Với quy mô rộng lớn, nội dung sâu sắc và toàn diện, nhịp điệu vô cùng nhanh chóng, cuộc cách mạng khoa học - kĩ thuật đã đưa lại biết bao thành tựu kì diệu và những đổi thay to lớn trong đời sống nhân loại. Nền văn minh thế giới có những bước nhảy vọt mới.'}, {'title': 'Nguồn gốc và đặc điểm của cuộc cách mạng